# Validation check for Step 00 - 16 result (v2)

In [2]:
from pathlib import Path
import json, pandas as pd, numpy as np

# ---------- Config ----------
AOI = "benguela"  # change if needed
# Primary project outputs path
B1 = Path("/mnt/c/Users/benny/OneDrive/Documents/Github/ago-lobitocorridor-analysis/outputs/tables")
# Fallback for ad-hoc uploads provided in this session
B2 = Path("/mnt/d/temp/wbg/iso3/ago/lobitocorridor/outputs/tables")

SEARCH_DIRS = [B1, B2]

def resolve(fname: str) -> Path:
    for base in SEARCH_DIRS:
        p = base / fname
        if p.exists():
            return p
    return B1 / fname

def ok(x): return "PASS" if x else "FAIL"
def exists_nonempty(p: Path) -> bool:
    try:
        return p.exists() and p.stat().st_size > 0
    except Exception:
        return False

def _check_v2_cols(df, v2_cols, label):
    """Check which v2 columns are present and report coverage."""
    present = sorted(c for c in v2_cols if c in df.columns)
    missing = sorted(c for c in v2_cols if c not in df.columns)
    if present:
        # Check non-null coverage for present v2 cols
        coverage = {c: f"{df[c].notna().mean():.0%}" for c in present}
        print(f"  v2 columns ({len(present)}/{len(v2_cols)}): {', '.join(present)}")
        print(f"  v2 coverage: {coverage}")
    if missing:
        print(f"  v2 missing ({len(missing)}): {', '.join(missing)}")

# ---------- Known / expected filenames ----------
paths = {
    # Core outputs
    "iso":        resolve(f"{AOI}_kpis_isochrones.csv"),
    "risk":       resolve(f"{AOI}_roads_flood_risk_summary.csv"),
    "muni":       resolve(f"{AOI}_municipality_indicators.csv"),
    "corr":       resolve(f"{AOI}_corr_with_rural_poverty.csv"),
    "prof":       resolve(f"{AOI}_municipality_profiles.csv"),
    "rank":       resolve(f"{AOI}_priority_muni_rank.csv"),
    "targeting":  resolve(f"{AOI}_muni_targeting.csv"),
    "scn_meta":   resolve(f"{AOI}_priority_scenarios.meta.json"),
    "scn_sum":    resolve(f"{AOI}_priority_scenarios_summary.csv"),
    "site":       resolve(f"{AOI}_site_audit_points.csv"),
    "proj":       resolve(f"{AOI}_project_kpis.csv"),
    "lookup":     resolve(f"{AOI}_admin2_lookup.csv"),
    "admin2_rank":resolve(f"{AOI}_priority_admin2_rank.csv"),
    "clusters":   resolve(f"{AOI}_priority_clusters.csv"),
    "catch_kpi":  resolve(f"{AOI}_catchments_kpis.csv"),
    "clust_syn":  resolve(f"{AOI}_cluster_synergies.csv"),
    "site_syn":   resolve(f"{AOI}_site_synergies.csv"),
    "od_grav":    resolve(f"{AOI}_od_gravity.csv"),
    "od_zone":    resolve(f"{AOI}_od_zone_attrs.csv"),
    "od_agents":  resolve(f"{AOI}_od_agents.csv"),
    # v2 outputs
    "marginal":   resolve(f"{AOI}_marginal_catchment.csv"),
    "dashboard":  resolve(f"corridor_dashboard.csv"),
    "sim":        resolve(f"{AOI}_sim_impact_summary.csv"),
}

print("=" * 80)
print(f"OUTPUT VALIDATION — AOI: {AOI}")
print("=" * 80)
print()
print("=== File presence check ===")
for k, p in paths.items():
    status = "FOUND" if p.exists() else "MISSING"
    size = f"({p.stat().st_size / 1024:.1f} KB)" if p.exists() else ""
    print(f"  {k:14s} -> {status:7s} {size}")
print()

# ==============================================================================
# 1) Isochrones (Step 02)
# ==============================================================================
try:
    p = paths["iso"]
    if exists_nonempty(p):
        iso = pd.read_csv(p)
        exp = {"aoi","travel_cut_min","pop_within","cells_within","area_km2_within"}
        print("[Isochrones]", ok(exp.issubset(iso.columns)), f"shape={iso.shape}")
        iso_s = iso.sort_values("travel_cut_min")
        mono_pop   = (iso_s["pop_within"].diff().fillna(0)  >= -1e-6).all()
        mono_cells = (iso_s["cells_within"].diff().fillna(0) >= -1e-6).all()
        print("  monotonic pop/cells:", ok(mono_pop and mono_cells))
        # v2 columns
        v2_iso = ["health_cover_60min_pct","tt_health_mean_within","building_count_within",
                   "dre_demand_within","drought_mean_within","gsl_mean_within"]
        _check_v2_cols(iso, v2_iso, "Isochrones")
        print(iso.head(3).to_string(index=False))
    else:
        print("[Isochrones] MISSING or EMPTY")
except Exception as e:
    print("[Isochrones] ERROR:", e)

# ==============================================================================
# 2) Roads × Flood
# ==============================================================================
try:
    p = paths["risk"]
    if exists_nonempty(p):
        risk = pd.read_csv(p)
        if not risk.empty:
            r = risk.iloc[0]
            def fnum(k):
                try: return float(r.get(k, np.nan))
                except: return np.nan
            ok_range = (0 <= fnum("risk_pct_of_roads") <= 100) and (0 <= fnum("near_prio_pct_of_risk") <= 100)
            print("[Roads×Flood] ranges:", ok(ok_range))
            for k in ["total_road_cells","total_risk_cells","risk_near_priority_cells"]:
                print(f"  {k}:", r.get(k, "<missing>"))
        else:
            print("[Roads×Flood] EMPTY")
    else:
        print("[Roads×Flood] MISSING or EMPTY")
except Exception as e:
    print("[Roads×Flood] ERROR:", e)

# ==============================================================================
# 3) Municipality indicators
# ==============================================================================
muni = None
try:
    p = paths["muni"]
    if exists_nonempty(p):
        muni = pd.read_csv(p)
        idc = [c for c in ["ADM2CD_c","NAM_1","NAM_2"] if c in muni.columns]
        n_unique = len(muni[idc].drop_duplicates()) if idc else None
        print("[Muni indicators]", ok(n_unique==len(muni)),
              f"rows={len(muni)} unique_adm2={n_unique}")
        num = muni.select_dtypes(include="number")
        na_mean = float(num.isna().mean().mean()) if not num.empty else np.nan
        print("  mean numeric NA share:", f"{na_mean:.2%}")
    else:
        print("[Muni indicators] MISSING or EMPTY")
except Exception as e:
    print("[Muni indicators] ERROR:", e)

# ==============================================================================
# 4) Correlations
# ==============================================================================
try:
    p = paths["corr"]
    if exists_nonempty(p):
        corr = pd.read_csv(p)
        if not corr.empty and {"theme","var","r","p","n"}.issubset(corr.columns):
            n_max = int(pd.to_numeric(corr["n"], errors="coerce").max())
            n_rows = len(muni) if muni is not None else None
            print("[Muni correlations] n<=#ADM2:", ok(n_rows is None or n_max <= n_rows),
                  f"n_max={n_max} #ADM2={n_rows}")
            top = corr.assign(absr=np.abs(pd.to_numeric(corr["r"], errors="coerce")))\
                      .sort_values("absr", ascending=False).head(5)
            print(top[["theme","var","r","p","n"]].to_string(index=False))
        else:
            print("[Muni correlations] present but missing cols or EMPTY")
    else:
        print("[Muni correlations] MISSING or EMPTY")
except Exception as e:
    print("[Muni correlations] ERROR:", e)

# ==============================================================================
# 5) Profiles
# ==============================================================================
try:
    p = paths["prof"]
    if exists_nonempty(p):
        prof = pd.read_csv(p)
        has_q = "poverty_quintile" in prof.columns
        print("[Profiles] quintile present:", ok(has_q), f"rows={len(prof)}")
        if has_q:
            print("  quintile counts:\n", prof["poverty_quintile"].value_counts(dropna=False))
    else:
        print("[Profiles] MISSING or EMPTY")
except Exception as e:
    print("[Profiles] ERROR:", e)

# ==============================================================================
# 6) Municipality shortlist / targeting (Step 09)
# ==============================================================================
try:
    # Try targeting first, fall back to rank
    p = paths["targeting"] if exists_nonempty(paths["targeting"]) else paths["rank"]
    label = "Muni targeting" if p == paths["targeting"] else "Muni shortlist"
    if exists_nonempty(p):
        rank = pd.read_csv(p)
        required = {
            "ADM2CD_c", "NAM_1", "NAM_2",
            "area_km2", "pop_total",
            "pop_le60min", "pop_le120min", "pop_gt120min",
            "cropland_km2", "pct_electrified", "pct_rural",
            "score",
        }
        has_required = required.issubset(rank.columns)
        print(f"[{label}] columns:", ok(has_required), f"shape={rank.shape}")
        if not has_required:
            missing = sorted(required - set(rank.columns))
            print("  missing required cols:", missing)

        # v2 columns for municipality targeting
        v2_muni = ["tt_health_mean","tt_edu_mean","building_density_mean",
                    "dre_demand_total","drought_events_mean","gsl_median_mean"]
        _check_v2_cols(rank, v2_muni, label)

        # 1 row per ADM2
        if "ADM2CD_c" in rank.columns:
            n_unique = rank["ADM2CD_c"].nunique()
            print(f"  unique ADM2CD_c: {ok(n_unique == len(rank))} rows={len(rank)} unique={n_unique}")

        # Score in [0,1]
        if "score" in rank.columns:
            smin, smax = float(rank["score"].min()), float(rank["score"].max())
            print(f"  score range [0-1]: {ok(0 <= smin and smax <= 1)} [{smin:.4f}, {smax:.4f}]")
    else:
        print(f"[{label}] MISSING or EMPTY")
except Exception as e:
    print("[Muni targeting] ERROR:", e)

# ==============================================================================
# 7) Scenarios meta/summary (Step 10)
# ==============================================================================
try:
    scn_ids_meta = []
    pm = paths["scn_meta"]
    if exists_nonempty(pm):
        meta = json.loads(pm.read_text())
        if isinstance(meta, list):
            scn_ids_meta = [d.get("id") for d in meta if isinstance(d, dict)]
            # v2: check for methodology fields
            has_agg = any(d.get("methodology",{}).get("aggregation") for d in meta if isinstance(d, dict))
            has_sel = any(d.get("methodology",{}).get("selection_method") for d in meta if isinstance(d, dict))
            print("[Scenarios meta] v2 methodology:", ok(has_agg), f"aggregation={has_agg} selection={has_sel}")
    ps = paths["scn_sum"]
    if exists_nonempty(ps):
        scn = pd.read_csv(ps)
        scn_ids_sum = sorted(scn["scenario_id"].unique()) if "scenario_id" in scn.columns else []
        scn_ok = not scn_ids_meta or (set(scn_ids_meta) == set(scn_ids_sum))
        print("[Scenarios summary] ids match meta:", ok(scn_ok), f"count={len(scn_ids_sum)}")
        for k in ["overlap_pct_vs_baseline","jaccard_vs_baseline","selected_cells","selected_km2"]:
            if k in scn.columns:
                print(f"  {k}: min={scn[k].min():.2f} mean={scn[k].mean():.2f} max={scn[k].max():.2f}")
        # v2 scenario metrics
        v2_scn = ["tt_health_mean_selected","tt_edu_mean_selected","bldg_dens_mean_selected","dre_demand_sum_selected"]
        _check_v2_cols(scn, v2_scn, "Scenarios")
    else:
        print("[Scenarios] MISSING or EMPTY")
except Exception as e:
    print("[Scenarios] ERROR:", e)

# ==============================================================================
# 8) Site audit points (Step 05)
# ==============================================================================
try:
    p = paths["site"]
    if exists_nonempty(p):
        site = pd.read_csv(p)
        cols = {c.lower() for c in site.columns}
        has_xy = any(c in cols for c in ["x","lon","longitude"]) and any(c in cols for c in ["y","lat","latitude"])
        print("[Site audit points] has XY:", ok(has_xy), f"shape={site.shape}")
        # v2 columns
        v2_site = ["tt_health_min","tt_education_min","building_density",
                    "gsl_median_days","spei12_drought_events"]
        _check_v2_cols(site, v2_site, "Site audit")
        # Coordinate sanity
        for c in ["lon","x","longitude"]:
            if c in site.columns:
                bad = (~site[c].between(-180, 180)).sum()
                if bad:
                    print(f"  WARNING: {bad} rows with {c} outside [-180,180]")
                break
    else:
        print("[Site audit points] MISSING or EMPTY")
except Exception as e:
    print("[Site audit points] ERROR:", e)

# ==============================================================================
# 9) Project KPIs (Step 08)
# ==============================================================================
try:
    p = paths["proj"]
    if exists_nonempty(p):
        proj = pd.read_csv(p)
        print("[Project KPIs] shape:", proj.shape)
        # v2 columns (check for any radius)
        v2_proj_patterns = ["tt_health_mean_","health_cover_60min_","tt_edu_mean_",
                            "bldg_dens_mean_","dre_demand_sum_","gsl_mean_"]
        v2_found = [c for c in proj.columns if any(c.startswith(pat) for pat in v2_proj_patterns)]
        if v2_found:
            print(f"  v2 columns ({len(v2_found)}): {', '.join(v2_found[:8])}{'...' if len(v2_found)>8 else ''}")
            # Check for nonsensical values (overflow from bad coords)
            for c in v2_found:
                vals = pd.to_numeric(proj[c], errors="coerce")
                if vals.notna().any():
                    vmax = vals.max()
                    if abs(vmax) > 1e30:
                        print(f"  WARNING: {c} has extreme values (max={vmax:.2e}) — possible bad coordinates")
        else:
            print("  v2 columns: NONE found")
        # Coordinate sanity
        if "lon" in proj.columns:
            bad = proj[~proj["lon"].between(-180, 180)]
            if len(bad):
                print(f"  WARNING: {len(bad)} rows with invalid lon (likely nodata site coordinates)")
    else:
        print("[Project KPIs] MISSING or EMPTY")
except Exception as e:
    print("[Project KPIs] ERROR:", e)

# ==============================================================================
# 10) Admin2 Lookup
# ==============================================================================
lookup = None
try:
    p = paths["lookup"]
    if exists_nonempty(p):
        lookup = pd.read_csv(p)
        need_cols = {"lab", "ADM2CD_c", "NAM_1", "NAM_2"}
        has_cols = need_cols.issubset(lookup.columns)
        print("[Admin2 Lookup] columns:", ok(has_cols), f"shape={lookup.shape}")
        if has_cols:
            is_unique = lookup["ADM2CD_c"].is_unique
            print("  unique ADM2CD_c:", ok(is_unique))
            if "lab" in lookup.columns:
                expected_labs = list(range(1, len(lookup) + 1))
                actual_labs = sorted(lookup["lab"].tolist())
                labs_sequential = (actual_labs == expected_labs)
                print("  sequential lab:", ok(labs_sequential))
            if "NAM_1" in lookup.columns:
                provinces = lookup["NAM_1"].unique()
                print(f"  provinces: {', '.join(provinces)}")
    else:
        print("[Admin2 Lookup] MISSING or EMPTY")
except Exception as e:
    print("[Admin2 Lookup] ERROR:", e)

# ==============================================================================
# 11) Priority Admin2 Rank
# ==============================================================================
admin2_rank = None
try:
    p = paths["admin2_rank"]
    if exists_nonempty(p):
        admin2_rank = pd.read_csv(p)
        need_cols = {"ADM2CD_c", "NAM_1", "NAM_2", "score", "rank", "selected", "share_selected"}
        has_cols = need_cols.issubset(admin2_rank.columns)
        print("[Priority Admin2 Rank] columns:", ok(has_cols), f"shape={admin2_rank.shape}")
        if has_cols:
            if admin2_rank["rank"].notna().any():
                rseq = sorted(admin2_rank["rank"].dropna().astype(int))
                contig = (rseq == list(range(min(rseq), max(rseq)+1)))
                print("  contiguous ranks:", ok(contig), f"range: {min(rseq)}-{max(rseq)}")
            if "score" in admin2_rank.columns:
                smin, smax = admin2_rank["score"].min(), admin2_rank["score"].max()
                print(f"  score range [0-1]: {ok(0 <= smin and smax <= 1)} [{smin:.4f}, {smax:.4f}]")
            if "selected" in admin2_rank.columns:
                n_sel = admin2_rank["selected"].sum()
                print(f"  selected: {n_sel}/{len(admin2_rank)} ({100*n_sel/len(admin2_rank):.1f}%)")
    else:
        print("[Priority Admin2 Rank] MISSING or EMPTY")
except Exception as e:
    print("[Priority Admin2 Rank] ERROR:", e)

# ==============================================================================
# 12) Priority clusters (Step 11)
# ==============================================================================
try:
    p = paths["clusters"]
    if exists_nonempty(p):
        cl = pd.read_csv(p)
        print("[Priority clusters] present:", ok(True), f"shape={cl.shape}")
        common = [c for c in ["cluster_id","cells","km2","score_mean","selected"] if c in cl.columns]
        if common:
            print("  core cols:", ", ".join(common))
            print(cl[common].head(5).to_string(index=False))
        # v2 columns
        v2_clust = ["tt_health_mean_min","tt_edu_mean_min","building_density_mean",
                     "dre_demand_total","gsl_median_days","spei12_events_mean"]
        _check_v2_cols(cl, v2_clust, "Clusters")
    else:
        print("[Priority clusters] MISSING or EMPTY")
except Exception as e:
    print("[Priority clusters] ERROR:", e)

# ==============================================================================
# 13) Catchments KPIs (Step 12)
# ==============================================================================
try:
    if exists_nonempty(paths["catch_kpi"]):
        ck = pd.read_csv(paths["catch_kpi"])
        need = {"site_index","thresh_min"}
        print("[Catchments KPIs] columns:", ok(need.issubset(ck.columns)), f"shape={ck.shape}")

        ck["thresh_min"] = pd.to_numeric(ck["thresh_min"], errors="coerce")
        if "area_km2" in ck.columns:
            ck["area_km2"] = pd.to_numeric(ck["area_km2"], errors="coerce")
            ck_sorted = ck.sort_values(["site_index","thresh_min"])
            mono_series = (
                ck_sorted
                .groupby("site_index", group_keys=False)["area_km2"]
                .apply(lambda s: (s.diff().fillna(0) >= -1e-6).all())
            )
            mono_pct = 100.0 * float(mono_series.mean()) if len(mono_series) else float("nan")
            print(f"  monotone area by site: {ok(bool(mono_series.all()))} | {mono_pct:.0f}% sites OK")

        # Coordinate sanity (catch the -1.79e308 bug)
        if "lon" in ck.columns:
            bad = ck[~ck["lon"].between(-180, 180)]
            if len(bad):
                print(f"  WARNING: {len(bad)} rows with invalid lon — site with nodata coords?")

        # v2 extra KPI columns (from preset EXTRA_KPI_RASTERS)
        extra_kpi = [c for c in ck.columns if c not in need and c not in
                     {"lon","lat","area_km2","pop","cropland_km2","rwi_mean","rwi_pop_weighted","mean_travel_min"}]
        if extra_kpi:
            print(f"  extra KPI cols ({len(extra_kpi)}): {', '.join(sorted(extra_kpi))}")
    else:
        print("[Catchments KPIs] MISSING or EMPTY")
except Exception as e:
    print("[Catchments KPIs] ERROR:", e)

# ==============================================================================
# 14) Synergies (clusters & sites)
# ==============================================================================
def _summarize_synergy(name, pth):
    try:
        if exists_nonempty(pth):
            df = pd.read_csv(pth)
            print(f"[{name}] present:", ok(True), f"shape={df.shape}")
            # Coordinate sanity
            for c in ["lon","lat"]:
                if c in df.columns:
                    vals = pd.to_numeric(df[c], errors="coerce")
                    bad = vals[~vals.between(-180 if c=="lon" else -90, 180 if c=="lon" else 90)]
                    if len(bad):
                        print(f"  WARNING: {len(bad)} rows with invalid {c}")
            print("  columns:", ", ".join(df.columns[:12]), "...")
        else:
            print(f"[{name}] MISSING or EMPTY")
    except Exception as e:
        print(f"[{name}] ERROR:", e)

_summarize_synergy("Cluster synergies", paths["clust_syn"])
_summarize_synergy("Site synergies", paths["site_syn"])

# ==============================================================================
# 15) OD-Lite
# ==============================================================================
try:
    pz = paths["od_zone"]
    grav = paths["od_grav"]
    ag = paths["od_agents"]

    if exists_nonempty(pz):
        Z = pd.read_csv(pz)
        has_xy = {"lon","lat"}.issubset(Z.columns)
        has_id = any(c in Z.columns for c in ["ADM2CD_c","adm2cd_c","id","lab"])
        print("[OD zone attrs] has lon/lat:", ok(has_xy), "| has zone id:", ok(has_id), f"shape={Z.shape}")
    else:
        Z = None
        print("[OD zone attrs] MISSING or EMPTY")

    if exists_nonempty(grav):
        G = pd.read_csv(grav)
        need = {"oi","dj","flow","dist_km"}
        print("[OD gravity] columns:", ok(need.issubset(G.columns)), f"rows={len(G)}")
        if need.issubset(G.columns):
            nonneg = (G["flow"] >= -1e-9).all()
            print("  non-negative flows:", ok(nonneg))
            total = G["flow"].sum()
            mean_d = np.average(G["dist_km"], weights=G["flow"]) if total > 0 else np.nan
            print(f"  total trips={total:,.0f} | flow-weighted mean dist={mean_d:,.1f} km")
    else:
        print("[OD gravity] MISSING or EMPTY")

    if exists_nonempty(ag):
        A = pd.read_csv(ag)
        need = {"oi","dj","o_lon","o_lat","d_lon","d_lat"}
        print("[OD agents] columns:", ok(need.issubset(A.columns)), f"N={len(A)}")
        for k in ["o_lon","d_lon"]:
            if k in A.columns:
                in_pct = A[k].between(-180, 180).mean()
                print(f"  {k} in [-180,180]: {in_pct:.2%}")
        for k in ["o_lat","d_lat"]:
            if k in A.columns:
                in_pct = A[k].between(-90, 90).mean()
                print(f"  {k} in [-90,90]: {in_pct:.2%}")
    else:
        print("[OD agents] MISSING or EMPTY")
except Exception as e:
    print("[OD] ERROR:", e)

# ==============================================================================
# 16) Simulation impact (Step 16)
# ==============================================================================
try:
    p = paths["sim"]
    if exists_nonempty(p):
        sim = pd.read_csv(p)
        print("[Simulation] present:", ok(True), f"shape={sim.shape}")
        for c in ["pop_gain_ge5min","pop_gain_ge10min","pop_gain_ge30min","pop_newly_le60min","pop_newly_le120min"]:
            if c in sim.columns:
                print(f"  {c}: {int(sim[c].iloc[0]):,}")
    else:
        print("[Simulation] MISSING or EMPTY")
except Exception as e:
    print("[Simulation] ERROR:", e)

# ==============================================================================
# CROSS-FILE VALIDATION
# ==============================================================================
print("\n" + "=" * 80)
print("CROSS-FILE VALIDATION")
print("=" * 80)

try:
    # Lookup ↔ Admin2 Rank
    if paths["lookup"].exists() and paths["admin2_rank"].exists():
        lookup = pd.read_csv(paths["lookup"])
        admin2_rank = pd.read_csv(paths["admin2_rank"])
        lookup_codes = set(lookup.get("ADM2CD_c", pd.Series(dtype=str)))
        rank_codes = set(admin2_rank.get("ADM2CD_c", pd.Series(dtype=str)))
        codes_match = (lookup_codes == rank_codes)
        print("[Lookup ↔ Admin2 Rank] ADM2CD_c match:", ok(codes_match))
        if not codes_match:
            missing_in_rank = sorted(list(lookup_codes - rank_codes))[:10]
            missing_in_lookup = sorted(list(rank_codes - lookup_codes))[:10]
            if missing_in_rank:
                print(f"  Missing in rank (first 10): {missing_in_rank}")
            if missing_in_lookup:
                print(f"  Missing in lookup (first 10): {missing_in_lookup}")

    # Lookup ↔ Muni Indicators count
    if paths["lookup"].exists() and paths["muni"].exists():
        lookup = pd.read_csv(paths["lookup"])
        muni = pd.read_csv(paths["muni"])
        n_lookup = len(lookup)
        n_muni_unique = len(muni[["ADM2CD_c"]].drop_duplicates()) if "ADM2CD_c" in muni.columns else None
        count_match = (n_muni_unique == n_lookup) if n_muni_unique is not None else False
        print("[Lookup ↔ Muni Indicators] count match:", ok(count_match),
              f"lookup={n_lookup} muni_unique={n_muni_unique}")

    # Admin2 Rank ↔ Muni shortlist
    if admin2_rank is not None and exists_nonempty(paths["rank"]):
        muni_rank = pd.read_csv(paths["rank"])
        if "ADM2CD_c" in admin2_rank.columns and "ADM2CD_c" in muni_rank.columns:
            codes_a = set(admin2_rank["ADM2CD_c"])
            codes_m = set(muni_rank["ADM2CD_c"])
            codes_match = (codes_a == codes_m)
            print("[Admin2 Rank ↔ Muni shortlist] ADM2CD_c match:", ok(codes_match))

    # OD zones ↔ lookup
    if paths["od_zone"].exists() and paths["lookup"].exists():
        Z = pd.read_csv(paths["od_zone"])
        L = pd.read_csv(paths["lookup"])
        z_id = None
        for cand in ["ADM2CD_c","adm2cd_c","lab","id"]:
            if cand in Z.columns:
                z_id = cand
                break
        if z_id is not None:
            n_match = len(set(Z[z_id])) == len(L)
            print("[OD zones ↔ Lookup] zone count matches:", ok(n_match), f"zones={len(Z)} lookup={len(L)}")

except Exception as e:
    print("[Cross-validation] ERROR:", e)

# ==============================================================================
# v2 LAYER COVERAGE SUMMARY
# ==============================================================================
print("\n" + "=" * 80)
print("v2 LAYER COVERAGE SUMMARY")
print("=" * 80)
v2_report = {
    "Site audit (Step 05)": (paths["site"], ["tt_health_min","tt_education_min","building_density","gsl_median_days","spei12_drought_events"]),
    "Project KPIs (Step 08)": (paths["proj"], ["tt_health_mean_5km","tt_edu_mean_5km","bldg_dens_mean_5km","dre_demand_sum_5km","gsl_mean_5km"]),
    "Muni targeting (Step 09)": (paths["targeting"] if exists_nonempty(paths["targeting"]) else paths["rank"],
                                  ["tt_health_mean","tt_edu_mean","building_density_mean","dre_demand_total","drought_events_mean","gsl_median_mean"]),
    "Scenarios (Step 10)": (paths["scn_sum"], ["tt_health_mean_selected","tt_edu_mean_selected","bldg_dens_mean_selected","dre_demand_sum_selected"]),
    "Clusters (Step 11)": (paths["clusters"], ["tt_health_mean_min","tt_edu_mean_min","building_density_mean","dre_demand_total","gsl_median_days","spei12_events_mean"]),
}
for label, (fp, expected) in v2_report.items():
    if exists_nonempty(fp):
        df = pd.read_csv(fp)
        found = sum(1 for c in expected if c in df.columns)
        pct = 100 * found / len(expected) if expected else 0
        status = "FULL" if found == len(expected) else f"PARTIAL ({found}/{len(expected)})" if found else "NONE"
        print(f"  {label:30s}: {status:20s} ({pct:.0f}%)")
    else:
        print(f"  {label:30s}: FILE MISSING")

print("\n" + "=" * 80)
print("VALIDATION COMPLETE")
print("=" * 80)

OUTPUT VALIDATION — AOI: benguela

=== File presence check ===
  iso            -> FOUND   (1.3 KB)
  risk           -> FOUND   (0.4 KB)
  muni           -> FOUND   (6.3 KB)
  corr           -> FOUND   (3.4 KB)
  prof           -> FOUND   (0.8 KB)
  rank           -> FOUND   (4.1 KB)
  targeting      -> MISSING 
  scn_meta       -> FOUND   (11.5 KB)
  scn_sum        -> FOUND   (5.1 KB)
  site           -> FOUND   (6.4 KB)
  proj           -> FOUND   (35.2 KB)
  lookup         -> FOUND   (0.3 KB)
  admin2_rank    -> FOUND   (1.3 KB)
  clusters       -> FOUND   (1.3 KB)
  catch_kpi      -> FOUND   (13.4 KB)
  clust_syn      -> FOUND   (0.5 KB)
  site_syn       -> FOUND   (1.3 KB)
  od_grav        -> FOUND   (5.4 KB)
  od_zone        -> FOUND   (1.1 KB)
  od_agents      -> FOUND   (8.0 KB)
  marginal       -> FOUND   (7.1 KB)
  dashboard      -> FOUND   (1.5 KB)
  sim            -> FOUND   (0.5 KB)

[Isochrones] PASS shape=(4, 21)
  monotonic pop/cells: PASS
  v2 columns (6/6): building_c